# K-Means sobre Word2Vec

Entrena Word2Vec, representa cada reseña por el promedio de sus vectores y la agrupa con K-Means, en la corrida de referencia y sobre diez semillas.

## Rutas

Las rutas son relativas a la carpeta `notebooks/`. Los datos no forman parte del repositorio (ver README).

In [ ]:
# Comprobación de rutas
import os
PROY = os.path.abspath('..')
print("carpeta del notebook :", os.getcwd())
print("raiz del proyecto    :", PROY)
for _sub in ("Topic/Dataframes", "Topic/Resultados", "csv_parques"):
    _p = os.path.join(PROY, _sub)
    print(f"  {'OK ' if os.path.isdir(_p) else 'FALTA'}  {_sub}")
print("\nSi algo dice FALTA, el kernel no arranco en la carpeta del notebook,\n"
      "o los datos no estan donde se los espera (ver README).")

### Corrida de referencia (Tablas 4.3 y 4.4)

In [ ]:
import pandas as pd
import numpy as np
import re
import multiprocessing
from tqdm import tqdm
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns

from gensim.models import Word2Vec
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score
from nltk.corpus import stopwords
import unicodedata
tqdm.pandas()

import re
import unicodedata
import spacy
from nltk.corpus import stopwords
import pickle

# --- Normalizador de tildes (MISMA función y nombre que LDA/TF-IDF) ---
def remove_accents(text):
    if not isinstance(text, str):
        return text
    return ''.join(c for c in unicodedata.normalize('NFD', text)
                   if unicodedata.category(c) != 'Mn')

# 1. CARGA + FILTRO DE IDIOMA (mismo corpus que LDA y TF-IDF)
df = pd.read_pickle("../df_generic_cleaned.pkl")

# 2. STOPWORDS — LA MISMA LISTA BASE QUE LDA (forma lematizada)
CONTEXTO = {
    'lugar', 'sitio', 'espacio', 'zona', 'parte', 'parque', 'montevideo', 'uruguay',
    'visita', 'visitar', 'paseo', 'pasear', 'ir', 'hacer', 'dia', 'noche',
    'fin', 'semana', 'año', 'vez', 'momento', 'cosa', 'caso', 'tema', 'tipo',
    'aunque', 'sino', 'pues', 'tal', 'tan', 'asi', 'ahi', 'alla', 'aca',
    'pais', 'ciudad', 'capital', 'gente', 'persona', 'publico', 'mate', 'pasar',
    'quedar', 'estar', 'ser', 'haber', 'tener', 'poder', 'deber',
    'dar', 'ver', 'saber', 'querer', 'llegar', 'decir', 'salir', 'venir',
    'encontrar', 'dejar', 'poner', 'parecer', 'llevar', 'seguir',
    'etc', 'siempre', 'alguno', 'solo', 'medio', 'ahora', 'mismo',
}
SENTIMIENTO = {
    'lindo', 'hermoso', 'bonito', 'bello', 'precioso',
    'feo', 'horrible', 'espantoso', 'terrible',
    'bueno', 'malo', 'buen',
    'excelente', 'espectacular', 'genial', 'fantastico', 'increible',
    'maravilloso', 'divino', 'impecable', 'impresionante',
    'pesimo', 'lamentable', 'desastre', 'patetico',
    'mejor', 'peor', 'mayor', 'menor', 'gran', 'grande',
    'bien', 'mal', 'super', 'muy', 'bastante', 'poco', 'mucho',
    'agradable', 'desagradable', 'recomendable', 'recomendado',
    'tranquilo', 'paz', 'relax',
    'gustar', 'encantar', 'disfrutar', 'recomendar',
    'valer', 'ideal', 'perfecto', 'destacar', 'pena', 'disfrutable',
}
ERRORES = {
    'lindar', 'exelente', 'exelentes', 'escelente',
    'imprecionante', 'ermoso', 'ermosa',
    'recomeir', 'recomiendable', 'recomendacion', 'mas',
}
CORTESIA = {'gracias'}
LUGARES  = {'prado'}

PROTEGER = {'nino', 'ninos', 'niño', 'niña', 'niñas', 'niños'}  # para evitar que "nino" se filtre como stopword
nombres_parque = set()
for nombre in df['nombre_parque'].str.lower().unique():
    for token in nombre.split():
        nombres_parque.add(remove_accents(token))
nombres_parque -= PROTEGER

final_stopwords = {remove_accents(w) for w in stopwords.words('spanish')}
for grupo in (CONTEXTO, SENTIMIENTO, ERRORES, CORTESIA, LUGARES):
    final_stopwords |= {remove_accents(w) for w in grupo}
final_stopwords |= nombres_parque
stop_set = set(final_stopwords)
print(f"Stopwords totales: {len(stop_set)}")

# 3. TOKENIZACIÓN + FILTRADO POR LEMA (el texto se entrena en superficie)
nlp = spacy.load("es_core_news_sm")
nlp.disable_pipes(["ner", "parser"])

def tokenizar_w2v(texto):
    if not isinstance(texto, str) or not texto.strip():
        return []
    texto = re.sub(r'[^a-záéíóúñü\s]', '', texto.lower())   # se conservan tildes PARA spaCy
    tokens = []
    for token in nlp(texto):
        lema  = remove_accents(token.lemma_.lower())
        forma = remove_accents(token.text.lower())
        if lema in stop_set or forma in stop_set:           # se tira por lema O por forma
            continue
        if len(forma) > 2 and not token.is_punct:
            tokens.append(forma)                            # entra la FORMA, no el lema
    return tokens

df['tokens_w2v'] = df['text'].progress_apply(tokenizar_w2v)
print("Ejemplo:")
print(f"  Antes:  {df['text'].iloc[0]}")
print(f"  Tokens: {df['tokens_w2v'].iloc[0]}")


# 4. ENTRENAR WORD2VEC

w2v_model = Word2Vec(
    sentences=df['tokens_w2v'],
    vector_size=100,
    window=5,
    min_count=5,
    sg=1,             # Skip-gram
    workers=1,
    seed=42,
    epochs=10         # explícito (default de gensim es 5)
)
print(f"Vocabulario: {len(w2v_model.wv.index_to_key)} palabras")


# 5. PRUEBAS DE SIMILITUD

def probar_modelo(modelo, palabra):
    try:
        print(f"\n--- Vecinos de '{palabra.upper()}' ---")
        vecinos = modelo.wv.most_similar(palabra, topn=5)
        for vecino, similitud in vecinos:
            print(f"  {vecino} ({round(similitud, 2)})")
    except KeyError:
        print(f"  Palabra '{palabra}' no está en el vocabulario")

probar_modelo(w2v_model, 'seguridad')
probar_modelo(w2v_model, 'banos')
probar_modelo(w2v_model, 'lindo')
probar_modelo(w2v_model, 'noche')
probar_modelo(w2v_model, 'juegos')
probar_modelo(w2v_model, 'ninos')


# 6. PCA 2D EXPLORATORIO

def plot_palabras(model, palabras):
    # filtrar palabras que estén en vocabulario
    palabras_validas = [p for p in palabras if p in model.wv]
    if len(palabras_validas) < 2:
        print("Menos de 2 palabras en vocabulario, no se puede graficar.")
        return
    vectors = [model.wv[p] for p in palabras_validas]
    pca = PCA(n_components=2)
    puntos = pca.fit_transform(vectors)

    plt.figure(figsize=(10, 7))
    for palabra, punto in zip(palabras_validas, puntos):
        plt.scatter(punto[0], punto[1], s=60)
        plt.text(punto[0] + 0.01, punto[1] + 0.01, palabra, fontsize=9)
    plt.title("PCA 2D de palabras clave (Word2Vec)")
    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("../Topic/Resultados/Topico/w2v_pca_palabras.png", dpi=300, bbox_inches='tight')
    plt.show()

plot_palabras(w2v_model, [
    "seguridad", "vigilancia", "tranquilo",
    "baños", "sucios", "limpios", "limpio",
    "noche", "iluminación", "oscuro",
    "hermoso", "precioso",
    "niños", "familia", "jugar"
])

# 7. ESTADÍSTICAS OOV

def oov_stats(tokens_series, model):
    total_tokens = 0
    oov = 0
    for tokens in tokens_series:
        total_tokens += len(tokens)
        for t in tokens:
            if t not in model.wv:
                oov += 1
    return {
        "total_tokens": total_tokens,
        "oov": oov,
        "oov_rate": round(oov / total_tokens * 100, 2)
    }

stats = oov_stats(df['tokens_w2v'], w2v_model)
print(f"OOV stats: {stats['oov']:,} / {stats['total_tokens']:,} = {stats['oov_rate']}%")


# 8. ZIPF 

tokens_list = [t for toks in df['tokens_w2v'] for t in toks]
cnt = Counter(tokens_list)
sorted_freqs = np.sort(np.array(list(cnt.values())))[::-1]
ranks = np.arange(1, len(sorted_freqs) + 1)

plt.figure(figsize=(7, 5))
plt.loglog(ranks, sorted_freqs, marker=".", linewidth=0, markersize=3)
plt.title("Rank-Frequency (Zipf) — Corpus W2V")
plt.xlabel("Rank")
plt.ylabel("Frecuencia")
plt.grid(True, which="both", ls=":", alpha=0.4)
plt.tight_layout()
plt.savefig("w2v_zipf.png", dpi=150, bbox_inches='tight')
plt.show()


# 9. VECTORIZACIÓN DE RESEÑAS (promedio de embeddings)

def obtener_vector_tematico(tokens, model, filtro):
    vectores = [model.wv[t] for t in tokens
                if t in model.wv.key_to_index and t not in filtro]
    if len(vectores) == 0:
        return None  # marcar como vacío en vez de vector cero
    return np.mean(vectores, axis=0)

df['vector_tema'] = df['tokens_w2v'].apply(
    lambda x: obtener_vector_tematico(x, w2v_model, stop_set)
)

# --- Filtrar reseñas sin vector ---
n_antes = len(df)
mask_nulo = df['vector_tema'].isna()
n_vacios = mask_nulo.sum()
print(f"Reseñas sin vector temático (todos los tokens filtrados): {n_vacios} / {n_antes} ({n_vacios/n_antes*100:.1f}%)")

df_valid = df[~mask_nulo].copy()
matrix_tema = np.vstack(df_valid['vector_tema'].values)
print(f"Reseñas para clustering: {len(df_valid)}")


# 10. KMEANS + BÚSQUEDA DE K ÓPTIMO
K_range = range(2, 10)
resultados_k = []

print("Evaluando K de 2 a 9...")
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=30)
    labels = km.fit_predict(matrix_tema)
    ch = calinski_harabasz_score(matrix_tema, labels)
    db = davies_bouldin_score(matrix_tema, labels)
    inertia = km.inertia_
    resultados_k.append({'K': k, 'CH': ch, 'DB': db, 'Inercia': inertia})
    print(f"  K={k}: CH={ch:.1f}, DB={db:.4f}, Inercia={inertia:.1f}")

df_metricas = pd.DataFrame(resultados_k)

# K óptimo por cada métrica
k_opt_ch = df_metricas.loc[df_metricas['CH'].idxmax(), 'K']
k_opt_db = df_metricas.loc[df_metricas['DB'].idxmin(), 'K']
print(f"\nK óptimo por Calinski-Harabasz: {k_opt_ch}")
print(f"K óptimo por Davies-Bouldin: {k_opt_db}")

# Gráfico CH vs DB
fig, ax1 = plt.subplots(figsize=(8, 5))
ax2 = ax1.twinx()

ax1.plot(df_metricas['K'], df_metricas['CH'], 'b-o', label='Calinski-Harabasz')
ax2.plot(df_metricas['K'], df_metricas['DB'], 'r--s', label='Davies-Bouldin')

ax1.axvline(x=6, color='gray', linestyle=':', alpha=0.7, label='K=6 (fijado)')
ax1.set_xlabel('Número de clusters (K)')
ax1.set_ylabel('Calinski-Harabasz (↑ mejor)', color='b')
ax2.set_ylabel('Davies-Bouldin (↓ mejor)', color='r')
ax1.legend(loc='upper left')
plt.title('Selección de K: Calinski-Harabasz vs Davies-Bouldin (W2V)')
plt.tight_layout()
plt.savefig("../Topic/Resultados/Topico/w2v_seleccion_k.png", dpi=150, bbox_inches='tight')
plt.show()


# 11. CLUSTERING FINAL CON K=6

K_FINAL = 6
print(f"Clustering final con K={K_FINAL}")

kmeans_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=30)
kmeans_final.fit(matrix_tema)
df_valid['cluster_w2v'] = kmeans_final.labels_

# Métricas finales
ch_final = calinski_harabasz_score(matrix_tema, kmeans_final.labels_)
db_final = davies_bouldin_score(matrix_tema, kmeans_final.labels_)
print(f"  Calinski-Harabasz: {ch_final:.1f}")
print(f"  Davies-Bouldin: {db_final:.4f}")
print(f"  Inercia: {kmeans_final.inertia_:.1f}")

# Distribución por cluster
print(f"\nDistribución:")
print(df_valid['cluster_w2v'].value_counts().sort_index())


# 12. PALABRAS POR CLUSTER — SIMILITUD COSENO AL CENTROIDE

print("TÓPICOS (palabras más cercanas al centroide de cada cluster)")

top_n_centroide = 15
cluster_keywords = {}

for i in range(K_FINAL):
    centroide = kmeans_final.cluster_centers_[i]
    palabras_cercanas = w2v_model.wv.most_similar(positive=[centroide], topn=50)
    # filtrar stopwords
    filtradas = [(p, sim) for p, sim in palabras_cercanas if p not in stop_set][:top_n_centroide]
    cluster_keywords[i] = filtradas
    words_str = ', '.join([f"{p} ({sim:.2f})" for p, sim in filtradas[:10]])
    print(f"\nCluster {i}: {words_str}")

# --- Gráfico: similitud coseno al centroide ---
fig, axes = plt.subplots(
    nrows=(K_FINAL + 1) // 2, ncols=2,
    figsize=(14, 4 * ((K_FINAL + 1) // 2)),
    constrained_layout=True
)
axes = axes.flatten()

for cluster_id in range(K_FINAL):
    ax = axes[cluster_id]
    kw = cluster_keywords[cluster_id]
    palabras = [p for p, s in kw]
    similitudes = [s for p, s in kw]

    ax.barh(palabras[::-1], similitudes[::-1], color='teal')
    ax.set_title(f'Cluster {cluster_id}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Similitud coseno al centroide')

# ocultar ejes sobrantes si K es impar
for j in range(K_FINAL, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Palabras clave por cluster — Similitud coseno (W2V + KMeans)',
             fontsize=14, fontweight='bold')
plt.savefig("../Topic/Resultados/Topico/W2V_keywords_por_cluster.png", dpi=150, bbox_inches='tight')
plt.show()


# 13. DISTRIBUCIÓN DE TÓPICOS POR PARQUE

perfil_parques = pd.crosstab(
    df_valid['nombre_parque'],
    df_valid['cluster_w2v'],
    normalize='index'
) * 100

print("Distribución de clusters por parque (%):")
display(perfil_parques.round(1))

# Heatmap 
plt.figure(figsize=(10, 8))
sns.heatmap(
    perfil_parques,
    annot=True, fmt=".1f",
    cmap="YlGnBu", linewidths=.5
)
plt.title('Intensidad de tópico por parque (W2V + KMeans)', fontsize=14)
plt.xlabel('Cluster')
plt.ylabel('Parque')
plt.tight_layout()
plt.savefig("../Topic/Resultados/Topico/W2V_heatmap_parques.png", dpi=150, bbox_inches='tight')
plt.show()

# --- Barras apiladas ---
ax = perfil_parques.plot(
    kind='barh', stacked=True,
    figsize=(12, 8), colormap='tab10', edgecolor='white'
)
plt.title('Distribución de clusters por parque (W2V + KMeans)', fontsize=14)
plt.xlabel('% de reseñas')
plt.ylabel('')
plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("../Topic/Resultados/Topico/W2V_barras_parques.png", dpi=150, bbox_inches='tight')
plt.show()


# 14. RESEÑAS REPRESENTATIVAS POR CLUSTER

from scipy.spatial.distance import cdist

print("RESEÑAS MÁS REPRESENTATIVAS (cercanas al centroide)")

for i in range(K_FINAL):
    mask_cluster = df_valid['cluster_w2v'] == i
    vecs_cluster = matrix_tema[mask_cluster.values]
    centroide = kmeans_final.cluster_centers_[i].reshape(1, -1)

    distancias = cdist(vecs_cluster, centroide, metric='cosine').flatten()
    idx_cercanos = distancias.argsort()[:3]

    df_cluster = df_valid[mask_cluster].iloc[idx_cercanos]
    print(f"Cluster {i} ({mask_cluster.sum()} reseñas) ---")
    for _, row in df_cluster.iterrows():
        texto = row['text'][:200]
        print(f"  • [{row['nombre_parque']}] {texto}")


# 15. RESUMEN DE MÉTRICA

print("RESUMEN")
print(f"K utilizado: {K_FINAL}")
print(f"K óptimo por Calinski-Harabasz: {k_opt_ch}")
print(f"K óptimo por Davies-Bouldin: {k_opt_db}")
print(f"Calinski-Harabasz en K={K_FINAL}: {ch_final:.1f}")
print(f"Davies-Bouldin en K={K_FINAL}: {db_final:.4f}")
print(f"Inercia: {kmeans_final.inertia_:.1f}")
print(f"Vocabulario W2V: {len(w2v_model.wv.index_to_key)} palabras")
print(f"Reseñas descartadas (sin vector): {n_vacios} / {n_antes}")
print(f"Random state: 42")


# 16. TOP TF-IDF POR PARQUE (opcional, análogo al de TF-IDF)



def top_words_por_parque_w2v(df_valid, matrix_tema, model, filtro, top_n=10):
    parques = df_valid['nombre_parque'].unique()
    resultados = {}

    for parque in sorted(parques):
        mask = (df_valid['nombre_parque'] == parque).values
        if mask.sum() == 0:
            continue
        centroide_parque = matrix_tema[mask].mean(axis=0)
        cercanas = model.wv.most_similar(positive=[centroide_parque], topn=50)
        filtradas = [(p, sim) for p, sim in cercanas if p not in filtro][:top_n]
        resultados[parque] = filtradas

    return resultados

top_kw_parques = top_words_por_parque_w2v(df_valid, matrix_tema, w2v_model, stop_set)
print("\nTop palabras por parque (W2V):")
for parque, kw in top_kw_parques.items():
    words = ', '.join([f"{p}" for p, s in kw[:7]])
    print(f"  {parque}: {words}")


df_valid[['id', 'nombre_parque', 'text', 'cluster_w2v']
].to_pickle("../Topic/Dataframes/df_w2v_final.pkl")


topic_words_raw = [[w for w, s in cluster_keywords[c]][:100] for c in sorted(cluster_keywords.keys())]
ref_texts = df_valid['tokens_w2v'].tolist()
pickle.dump({'topic_words_raw': topic_words_raw, 'ref_texts': ref_texts, 'final_stopwords': final_stopwords},
            open("../Topic/Dataframes/ckpt_w2v.pkl", "wb"))
print("ckpt_w2v.pkl OK")

### Coherencia C_v de la corrida de referencia

In [ ]:
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

# Corpus tokenizado (sin filtrar stopwords, para que C_v tenga contexto)
textos_tokenizados = df_valid['tokens_w2v'].tolist()

diccionario = Dictionary(textos_tokenizados)

# Top words por cluster
topic_words = []
for cluster_id in sorted(cluster_keywords.keys()):
    words = [w for w, sim in cluster_keywords[cluster_id][:15]]
    topic_words.append(words)

cm = CoherenceModel(
    topics=topic_words,
    texts=textos_tokenizados,
    dictionary=diccionario,
    coherence='c_v'
)

cv_score = cm.get_coherence()
print(f"Coherencia C_v (K-Means + Word2Vec): {cv_score:.4f}")

cv_por_topico = cm.get_coherence_per_topic()
for i, cv_t in enumerate(cv_por_topico):
    print(f"  Cluster {i}: C_v = {cv_t:.4f}")

### Coherencia C_v sobre las palabras filtradas

In [ ]:
# === Coherencia C_v — K-Means + Word2Vec ===
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

TOP_N = 15
topic_words = [
    [w for w, sim in cluster_keywords[c] if w not in final_stopwords][:TOP_N]
    for c in sorted(cluster_keywords.keys())
]

textos_tokenizados = df_valid['tokens_w2v'].tolist()
diccionario = Dictionary(textos_tokenizados)

cm = CoherenceModel(topics=topic_words, texts=textos_tokenizados,
                    dictionary=diccionario, coherence='c_v')
print(f"Coherencia C_v (K-Means + Word2Vec): {cm.get_coherence():.4f}")
for i, s in enumerate(cm.get_coherence_per_topic()):
    print(f"  Cluster {i}: {s:.4f}")

### Diez semillas y barrido de K (Tablas 3.2, 4.1 y 4.2)

In [ ]:
# ============================================================
# MULTI-SEMILLA — K-Means + Word2Vec  (autocontenido)
# Por semilla: RE-ENTRENA el embedding (Antoniak & Mimno 2018:
# el embedding es fuente de varianza dominante) UNA vez, y con ese mismo embedding hace el
# barrido K=2..9 (CH/DB) + top-100 en K=6. workers=1 -> reproducible por semilla.
# Cachea la tokenizacion (superficie) en prep_w2v_tokens.pkl.
# ============================================================
import pickle, os, re, unicodedata, numpy as np, pandas as pd
from gensim.models import Word2Vec
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score
from nltk.corpus import stopwords
import nltk, spacy
from tqdm import tqdm
nltk.download('stopwords', quiet=True)

BD    = ".."
DFDIR = f"{BD}/Topic/Dataframes"
SEEDS = [42, 0, 1, 2, 3, 4, 5, 6, 7, 8]
K_RANGE = range(2, 10)
K_FINAL = 6

def remove_accents(text):
    if not isinstance(text, str): return text
    return ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')

PREP = f"{DFDIR}/prep_w2v_tokens.pkl"
if os.path.exists(PREP):
    d = pickle.load(open(PREP, 'rb')); tokens = d['tokens']; final_stopwords = d['final_stopwords']
    stop_set = set(final_stopwords)
    print(f"prep_w2v_tokens.pkl cargado | {len(tokens)} docs")
else:
    df = pd.read_pickle(f"{BD}/df_generic_cleaned.pkl")
    CONTEXTO = {
        'lugar','sitio','espacio','zona','parte','parque','montevideo','uruguay',
        'visita','visitar','paseo','pasear','ir','hacer','dia','noche',
        'fin','semana','año','vez','momento','cosa','caso','tema','tipo',
        'aunque','sino','pues','tal','tan','asi','ahi','alla','aca',
        'pais','ciudad','capital','gente','persona','publico','mate','pasar',
        'quedar','estar','ser','haber','tener','poder','deber',
        'dar','ver','saber','querer','llegar','decir','salir','venir',
        'encontrar','dejar','poner','parecer','llevar','seguir',
        'etc','siempre','alguno','solo','medio','ahora','mismo'}
    SENTIMIENTO = {
        'lindo','hermoso','bonito','bello','precioso','feo','horrible','espantoso','terrible',
        'bueno','malo','buen','excelente','espectacular','genial','fantastico','increible',
        'maravilloso','divino','impecable','impresionante','pesimo','lamentable','desastre','patetico',
        'mejor','peor','mayor','menor','gran','grande','bien','mal','super','muy','bastante','poco','mucho',
        'agradable','desagradable','recomendable','recomendado','tranquilo','paz','relax',
        'gustar','encantar','disfrutar','recomendar','valer','ideal','perfecto','destacar','pena','disfrutable'}
    ERRORES = {'lindar','exelente','exelentes','escelente','imprecionante','ermoso','ermosa',
        'recomeir','recomiendable','recomendacion','mas'}
    CORTESIA = {'gracias'}; LUGARES = {'prado'}
    PROTEGER = {'nino','ninos','niño','niña','niñas','niños'}
    nombres = set()
    for nombre in df['nombre_parque'].str.lower().unique():
        for t in nombre.split(): nombres.add(remove_accents(t))
    nombres -= PROTEGER
    final_stopwords = {remove_accents(w) for w in stopwords.words('spanish')}
    for g in (CONTEXTO, SENTIMIENTO, ERRORES, CORTESIA, LUGARES):
        final_stopwords |= {remove_accents(w) for w in g}
    final_stopwords |= nombres
    stop_set = set(final_stopwords)
    nlp = spacy.load("es_core_news_sm"); nlp.disable_pipes(["ner", "parser"])
    def tok_w2v(t):
        if not isinstance(t, str) or not t.strip(): return []
        t = re.sub(r'[^a-záéíóúñü\s]', '', t.lower())
        out = []
        for tk in nlp(t):
            lema = remove_accents(tk.lemma_.lower()); forma = remove_accents(tk.text.lower())
            if lema in stop_set or forma in stop_set: continue
            if len(forma) > 2 and not tk.is_punct: out.append(forma)
        return out
    tqdm.pandas(desc="Tokenizando")
    tokens = df['text'].progress_apply(tok_w2v).tolist()
    final_stopwords = sorted(final_stopwords)
    pickle.dump({'tokens': tokens, 'final_stopwords': final_stopwords}, open(PREP, 'wb'))
    print(f"prep_w2v_tokens.pkl creado | {len(tokens)} docs")

def vec_tema(toks, model):
    v = [model.wv[w] for w in toks if w in model.wv.key_to_index and w not in stop_set]
    return np.mean(v, axis=0) if v else None

barrido = {}          # seed -> {k: (CH, DB)}
topw_by_seed = {}
ref_texts = None
for s in SEEDS:
    w2v = Word2Vec(sentences=tokens, vector_size=100, window=5, min_count=5,
                   sg=1, workers=1, seed=s, epochs=10)
    vecs = [vec_tema(t, w2v) for t in tokens]
    idx_valid = [i for i, v in enumerate(vecs) if v is not None]
    M = np.vstack([vecs[i] for i in idx_valid])
    if ref_texts is None:
        ref_texts = [tokens[i] for i in idx_valid]   # vocab W2V no depende de seed -> valido invariante
    curva = {}
    for k in K_RANGE:
        km = KMeans(n_clusters=k, random_state=s, n_init=30)
        lab = km.fit_predict(M)
        curva[k] = (float(calinski_harabasz_score(M, lab)), float(davies_bouldin_score(M, lab)))
        if k == K_FINAL:
            km6 = KMeans(n_clusters=K_FINAL, random_state=s, n_init=30); km6.fit(M)
            tw = []
            for i in range(K_FINAL):
                cerc = w2v.wv.most_similar(positive=[km6.cluster_centers_[i]], topn=200)
                tw.append([p for p, sim in cerc if p not in stop_set][:100])
            topw_by_seed[s] = tw
    barrido[s] = curva
    kbest = max(curva, key=lambda kk: curva[kk][0])
    print(f"W2V seed={s:>2} | vocab={len(w2v.wv.index_to_key)} valid={len(idx_valid)} | CH argmax=K{kbest}")

pickle.dump({'topic_words_raw_by_seed': topw_by_seed, 'ref_texts': ref_texts,
             'final_stopwords': sorted(stop_set), 'seeds': SEEDS},
            open(f"{DFDIR}/ckpt_w2v_multiseed.pkl", "wb"))
pickle.dump({'barrido': barrido, 'criterio': 'CH,DB', 'k_range': list(K_RANGE), 'seeds': SEEDS},
            open(f"{DFDIR}/barrido_w2v_multiseed.pkl", "wb"))
print("OK -> ckpt_w2v_multiseed.pkl + barrido_w2v_multiseed.pkl")